# DMPBridge Narrative Structure Extraction Strategy

## Workflow

```text
PDF
    ↓
PDFPlumber Extraction
    ↓
Text and Layout Metadata Extraction
    ↓
Block Cleaning and Normalization
    ↓
Layout-Aware Block Representation
    ↓
Chunk Construction with Context Overlap
    ↓
Llama 3.1 8B Narrative Labeling
    ↓
Original Text Reattachment
    ↓
Chunk-Level Label Aggregation
    ↓
Post-Processing and Validation
    ↓
Structured Narrative JSON
```

---

## 1. PDF

Input Data Management Plan (DMP) document in PDF format.

---

## 2. PDFPlumber Extraction

The PDF is processed using PDFPlumber to extract original text blocks while preserving document layout information.

Example extracted attributes:

- text
- page
- line_order
- font_size
- font_name
- is_bold
- x0
- x1
- top
- bottom

The extracted `text` field is the source of the final narrative text.

---

## 3. Text and Layout Metadata Extraction

Additional metadata are derived from the PDFPlumber output to provide contextual information about each block.

Examples include:

- relative_font
- indent
- vertical_gap_before
- vertical_gap_after
- position_hint
- word_count

These features describe how a block appears relative to the surrounding document structure.

---

## 4. Block Cleaning and Normalization

Extracted blocks are cleaned while preserving the original document wording.

Typical operations include:

- whitespace normalization
- repeated-word cleanup
- removal of empty blocks
- removal of extraction artifacts

This step improves consistency but does not generate new content.

---

## 5. Layout-Aware Block Representation

Each block is converted into a structured representation combining textual, visual, and positional information.

Example:

```json
{
  "block_id": 15,
  "text": "Data Format Standards",
  "page": 2,
  "font_size": 14,
  "is_bold": true,
  "indent": 72,
  "vertical_gap_before": 20
}
```

This representation serves as the input to the language model.

---

## 6. Chunk Construction with Context Overlap

To support efficient processing with Llama 3.1 8B, the document is divided into overlapping chunks.

Example:

```text
Chunk 1: Blocks 1–45
Chunk 2: Blocks 41–85
Chunk 3: Blocks 81–125
```

Each chunk contains:

- target blocks to be labeled
- neighboring context blocks before and after the target region

The overlap preserves continuity across chunk boundaries.

---

## 7. Llama 3.1 8B Narrative Labeling

For each chunk, the model analyzes:

- textual content
- typography
- spacing
- indentation
- document position
- neighboring context

The model assigns one of the following labels to each target block:

- document_title
- section
- subsection
- content

The model returns only block IDs and labels.

---

## 8. Original Text Reattachment

After labeling, Python reconnects each LLM-assigned label to the original PDFPlumber-extracted text.

Example:

```json
{
  "label": "section",
  "text": "Data Format Standards"
}
```

In this output:

- `label` comes from Llama
- `text` comes from PDFPlumber

This keeps the final structured JSON grounded in the original extracted document content.

---

## 9. Chunk-Level Label Aggregation

Labels generated from all chunks are merged into a single document-level structure.

This stage:

- combines chunk outputs
- resolves overlapping regions
- preserves original block order

---

## 10. Post-Processing and Validation

Lightweight refinement is applied to improve consistency.

Tasks include:

- label validation
- merging consecutive title blocks
- merging consecutive content blocks
- generation of diagnostic summaries

This step does not create new narrative content.

---

## 11. Structured Narrative JSON

The final output is a structured narrative representation of the DMP.

Example:

```json
[
  {
    "label": "document_title",
    "text": "CAREER: HIGH-RESOLUTION NMR FOR PARAMAGNETIC SODIUM ELECTRODES"
  },
  {
    "label": "section",
    "text": "Data Format Standards"
  },
  {
    "label": "content",
    "text": "The project will generate..."
  }
]
```

---

## Summary

DMPBridge converts PDFPlumber-extracted text and layout metadata into a layout-aware block representation, applies chunked Llama 3.1 8B narrative labeling, reattaches the original extracted text to the assigned labels, and produces a structured JSON representation of the document hierarchy.

The LLM functions as a block-level label classifier, while the final text remains grounded in the original PDFPlumber extraction.


## Project root setup

In [1]:
from pathlib import Path
import json
import importlib
import sys
import traceback

# Project root setup first
cwd = Path.cwd()

if (cwd / "data").exists() and (cwd / "src").exists():
    project_root = cwd
else:
    project_root = cwd.parent

src_path = str(project_root / "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("Project root:", project_root)
print("Source path:", src_path)

Project root: c:\Users\Nahid\dmpbridge
Source path: c:\Users\Nahid\dmpbridge\src


In [2]:
from dmpbridge.llm.llama_client import load_llama
import dmpbridge.llm.llm_narrative_blocks_plumberjson as lnb
from dmpbridge.processing.text_cleaner import clean_repeated_words

# Reload module to avoid old cached notebook version
importlib.reload(lnb)

generate_structured_blocks_with_llm = lnb.generate_structured_blocks_with_llm
save_blocks = lnb.save_blocks

print("LLM narrative blocks module loaded.")

LLM narrative blocks module loaded.


In [3]:
pdfplumber_blocks_dir = (
    project_root
    / "data"
    / "pdfplumber_extracted_blocks"
)

llama_output_dir = (
    project_root
    / "data"
    / "llama_structured_blocks"
)

llama_output_dir.mkdir(parents=True, exist_ok=True)

print("PDFPlumber blocks directory:", pdfplumber_blocks_dir)
print("Llama output directory:", llama_output_dir)

PDFPlumber blocks directory: c:\Users\Nahid\dmpbridge\data\pdfplumber_extracted_blocks
Llama output directory: c:\Users\Nahid\dmpbridge\data\llama_structured_blocks


In [4]:
llm = load_llama(
    model_name="llama3.1:8b",
    temperature=0,
)

print("Llama loaded successfully.")

Llama loaded successfully.


# Process all PDFPlumber block JSON files

In [5]:
pdfplumber_block_files = sorted(
    pdfplumber_blocks_dir.glob("*.json")
)

print(f"\nFound {len(pdfplumber_block_files)} PDFPlumber block files")


for block_path in pdfplumber_block_files:
    sample_name = block_path.stem

    print("\n" + "=" * 80)
    print(f"Processing: {sample_name}")
    print("=" * 80)

    try:
        # Load PDFPlumber blocks
        # ----------------------------------------------------

        with open(block_path, "r", encoding="utf-8") as f:
            pdfplumber_blocks = json.load(f)

        print(f"Original PDFPlumber blocks: {len(pdfplumber_blocks)}")

        # Clean repeated words before LLM structure extraction
        # ----------------------------------------------------

        cleaned_pdfplumber_blocks = []

        for block in pdfplumber_blocks:
            if not isinstance(block, dict):
                continue

            cleaned_block = dict(block)

            cleaned_text = clean_repeated_words(
                str(block.get("text", ""))
            ).strip()

            if not cleaned_text:
                continue

            cleaned_block["text"] = cleaned_text
            cleaned_pdfplumber_blocks.append(cleaned_block)

        print(f"Cleaned PDFPlumber blocks: {len(cleaned_pdfplumber_blocks)}")

        if not cleaned_pdfplumber_blocks:
            print("Skipped because no valid text blocks were found.")
            continue

        # Main: LLM-only structure extraction
        # ----------------------------------------------------

        llama_blocks = generate_structured_blocks_with_llm(
            llm=llm,
            pdf_blocks=cleaned_pdfplumber_blocks,
        )

        llama_output_path = (
            llama_output_dir
            / f"{sample_name}_llama_blocks.json"
        )

        save_blocks(
            blocks=llama_blocks,
            output_path=llama_output_path,
        )

        print(
            f"Saved Llama structured blocks: {len(llama_blocks)} -> "
            f"{llama_output_path.name}"
        )

    except Exception as e:
        print(f"ERROR processing {sample_name}")
        print(type(e).__name__, ":", e)
        traceback.print_exc()


print("\nFinished processing all PDFPlumber block files.")


Found 10 PDFPlumber block files

Processing: sample1
Original PDFPlumber blocks: 79
Cleaned PDFPlumber blocks: 79
Saved Llama structured blocks: 28 -> sample1_llama_blocks.json

Processing: sample10
Original PDFPlumber blocks: 68
Cleaned PDFPlumber blocks: 68
Saved Llama structured blocks: 13 -> sample10_llama_blocks.json

Processing: sample2
Original PDFPlumber blocks: 171
Cleaned PDFPlumber blocks: 171
Saved Llama structured blocks: 20 -> sample2_llama_blocks.json

Processing: sample3
Original PDFPlumber blocks: 69
Cleaned PDFPlumber blocks: 69
Saved Llama structured blocks: 11 -> sample3_llama_blocks.json

Processing: sample4
Original PDFPlumber blocks: 78
Cleaned PDFPlumber blocks: 78
Saved Llama structured blocks: 29 -> sample4_llama_blocks.json

Processing: sample5
Original PDFPlumber blocks: 81
Cleaned PDFPlumber blocks: 81
Saved Llama structured blocks: 31 -> sample5_llama_blocks.json

Processing: sample6
Original PDFPlumber blocks: 24
Cleaned PDFPlumber blocks: 24
Saved Llama